In [1]:
!pip install -q -U \
    "transformers>=4.45.0" \
    "accelerate>=0.34.0" \
    "gradio==5.49.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-fasthtml 0.14.6 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.4.0 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is inco

In [ ]:
import warnings

import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as transformers_logging


# ---------------------------------------------------------
# 1. Configurações iniciais
# ---------------------------------------------------------

warnings.filterwarnings("ignore")
transformers_logging.set_verbosity_error()

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


# ---------------------------------------------------------
# 2. Carregamento do tokenizer
# ---------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ---------------------------------------------------------
# 3. Carregamento do modelo
# ---------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Modelo carregado com sucesso!")
print("Versão do Gradio:", gr.__version__)


# ---------------------------------------------------------
# 4. Função que responde ao usuário
# ---------------------------------------------------------

def responder(mensagem, historico):
    """
    Recebe:
    - mensagem: nova pergunta do usuário;
    - historico: mensagens anteriores administradas pelo Gradio.

    Retorna:
    - somente a nova resposta produzida pelo modelo.
    """

    if not mensagem or not mensagem.strip():
        return "Digite uma pergunta."

    mensagens = [
        {
            "role": "system",
            "content": (
                "Você é um assistente educacional. "
                "Responda sempre em português brasileiro, "
                "de forma clara, simples e objetiva."
            )
        }
    ]

    # O histórico do Gradio chega no formato:
    # {"role": "user", "content": "..."}
    # {"role": "assistant", "content": "..."}
    for item in historico:
        if not isinstance(item, dict):
            continue

        role = item.get("role")
        content = item.get("content")

        # Ignora itens que não sejam mensagens textuais válidas
        if role not in {"user", "assistant"}:
            continue

        if not isinstance(content, str):
            continue

        mensagens.append(
            {
                "role": role,
                "content": content
            }
        )

    # Adiciona a pergunta atual
    mensagens.append(
        {
            "role": "user",
            "content": mensagem.strip()
        }
    )

    # Converte a conversa para o formato esperado pelo Qwen
    inputs = tokenizer.apply_chat_template(
        mensagens,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move cada tensor para o mesmo dispositivo do modelo
    inputs = {
        nome: tensor.to(model.device)
        for nome, tensor in inputs.items()
    }

    # Geração da resposta
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Remove os tokens que pertencem ao prompt
    quantidade_tokens_entrada = inputs["input_ids"].shape[1]

    novos_tokens = outputs[0][quantidade_tokens_entrada:]

    # Transforma os novos tokens em texto
    resposta = tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

    return resposta.strip()


# ---------------------------------------------------------
# 5. Componente visual do chat
# ---------------------------------------------------------

chatbot = gr.Chatbot(
    type="messages",
    height=600,
    min_width=300,
    placeholder="A conversa aparecerá aqui."
)


# ---------------------------------------------------------
# 6. Interface completa
# ---------------------------------------------------------

aplicacao = gr.ChatInterface(
    fn=responder,

    chatbot=chatbot,

    textbox=gr.Textbox(
        placeholder="Digite sua pergunta...",
        lines=1,
        container=False
    ),

    type="messages",

    title="Assistente de Inteligência Artificial",

    description=(
        "Chat simples executado com um modelo Transformer "
        "diretamente no Google Colab."
    ),

    examples=[
        "O que é inteligência artificial?",
        "Explique o que é um Transformer.",
        "O que significa tokenização?"
    ]
)


# ---------------------------------------------------------
# 7. Inicialização
# ---------------------------------------------------------

aplicacao.launch(
    share=True,
    debug=True
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado com sucesso!
Versão do Gradio: 5.49.1
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://128d3f5f2f994c829d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
